# 🎬 콘텐츠 기반 영화 추천 시스템

> 참고 도서: 『파이썬 머신러닝 완벽 가이드』 9장 - 추천 시스템

이 노트북에서는 TMDB 5000 Movie Dataset을 사용해서, 장르/키워드/감독/배우 정보를 바탕으로
'비슷한 영화'를 추천해주는 콘텐츠 기반 필터링(Content-based Filtering) 추천 시스템을 만듭니다.

## 진행 단계
1~6. 데이터 로딩 및 가공 (라이브러리 확인, CSV 로딩, JSON 파싱, 감독 추출, merge, EDA)
7~9. 장르 벡터화(CountVectorizer) → 코사인 유사도 계산 → 유사 영화 추천 함수 구현
10~12. 가중 평점(Weighted Rating) 반영 → Before/After 비교
13~14. TMDB 포스터 조회 → 웹 UI용 데이터(`app/data.json`) export

> 이 노트북 실행이 끝나면 `app/recommender_ui.html`에서 바로 시연할 수 있는 데이터가 준비됩니다.

## 1. 라이브러리 불러오기

이 프로젝트에서 사용할 라이브러리들입니다.

- `pandas`: 표(테이블) 형태의 데이터를 다루는 라이브러리 (엑셀을 코드로 다룬다고 생각하면 쉬워요)
- `numpy`: 숫자 계산(배열, 행렬 연산 등)을 도와주는 라이브러리
- `ast`: 파이썬 표준 라이브러리. 문자열로 되어있는 리스트/딕셔너리를 진짜 파이썬 객체로 바꿔줍니다
- `scikit-learn`: 다음 단계(벡터화, 코사인 유사도)에서 사용할 머신러닝 라이브러리 (지금은 설치 확인만)
- `python-dotenv`, `requests`: 나중에 TMDB API로 포스터 이미지를 가져올 때 사용 (지금은 설치 확인만)

In [1]:
# 데이터 처리를 위한 기본 라이브러리
import pandas as pd
import numpy as np

# 문자열 형태의 JSON(리스트/딕셔너리)을 진짜 파이썬 객체로 변환할 때 사용
import ast

# 아래 라이브러리들은 이번 단계에서 직접 사용하지는 않지만,
# 프로젝트 전체(다음 단계 포함)에서 필요하므로 설치가 잘 되어 있는지 미리 확인만 합니다.
import sklearn
import dotenv
import requests

# 각 라이브러리의 버전을 출력해서, 설치가 정상적으로 되었는지 확인합니다.
print(f"pandas 버전: {pd.__version__}")
print(f"numpy 버전: {np.__version__}")
print(f"scikit-learn 버전: {sklearn.__version__}")
print(f"requests 버전: {requests.__version__}")
print("모든 라이브러리를 정상적으로 불러왔습니다 ✅")

pandas 버전: 3.0.5
numpy 버전: 2.5.2
scikit-learn 버전: 1.9.0
requests 버전: 2.34.2
모든 라이브러리를 정상적으로 불러왔습니다 ✅


## 2. 데이터 불러오기

`data/` 폴더 안에 있는 두 개의 CSV 파일을 불러옵니다.

- `tmdb_5000_movies.csv`: 영화의 장르, 줄거리, 평점 등 기본 정보
- `tmdb_5000_credits.csv`: 영화별 출연진(cast)과 제작진(crew) 정보

In [2]:
# 노트북 파일(movie_recommender.ipynb)은 notebook/ 폴더 안에 있으므로,
# 상위 폴더(..)의 data/ 폴더 경로를 지정해줍니다.
MOVIES_PATH = "../data/tmdb_5000_movies.csv"
CREDITS_PATH = "../data/tmdb_5000_credits.csv"

movies = pd.read_csv(MOVIES_PATH)
credits = pd.read_csv(CREDITS_PATH)

print(f"movies shape: {movies.shape}")
print(f"credits shape: {credits.shape}")

movies shape: (4803, 20)
credits shape: (4803, 4)


In [3]:
# movies 데이터의 앞부분 5개 행을 확인합니다.
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [4]:
# credits 데이터의 앞부분 5개 행을 확인합니다.
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## 3. JSON 문자열 컬럼 파싱하기

`genres`, `keywords`, `cast`, `crew` 컬럼은 실제로는 리스트/딕셔너리인데,
CSV 파일로 저장되면서 **문자열(string)** 형태로 바뀌어 있습니다. 예를 들면 이렇게 생겼어요:

```
"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]"
```

이 상태로는 파이썬이 이걸 '리스트 안에 딕셔너리가 있는 것'으로 인식하지 못하고,
그냥 하나의 긴 문자열로만 취급합니다. 그래서 `ast.literal_eval()` 함수를 사용해서
**문자열을 실제 파이썬 리스트/딕셔너리로 변환**해줘야 합니다.

> `ast.literal_eval`은 `eval()`과 비슷하지만, 파이썬 리터럴(문자열, 숫자, 리스트, 딕셔너리 등)만
> 안전하게 변환해주기 때문에 `eval()`보다 안전합니다.

In [5]:
# 변환 전, genres 컬럼의 첫 번째 값이 어떻게 생겼는지 타입과 함께 확인해봅니다.
print(type(movies.loc[0, "genres"]))
print(movies.loc[0, "genres"])

<class 'str'>
[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]


In [6]:
# movies 데이터에서 JSON 문자열로 되어 있는 컬럼들
movies_json_columns = ["genres", "keywords", "production_companies", "production_countries", "spoken_languages"]

# credits 데이터에서 JSON 문자열로 되어 있는 컬럼들
credits_json_columns = ["cast", "crew"]

# apply(ast.literal_eval)을 사용하면 컬럼의 모든 행에 대해
# '문자열 -> 실제 파이썬 객체' 변환을 한 번에 적용할 수 있습니다.
for col in movies_json_columns:
    movies[col] = movies[col].apply(ast.literal_eval)

for col in credits_json_columns:
    credits[col] = credits[col].apply(ast.literal_eval)

print("JSON 문자열 파싱 완료 ✅")

JSON 문자열 파싱 완료 ✅


In [7]:
# 변환 후, genres 컬럼의 첫 번째 값이 어떻게 바뀌었는지 확인합니다.
# 이제 문자열이 아니라 '리스트 안에 딕셔너리가 담긴' 진짜 파이썬 객체입니다.
print(type(movies.loc[0, "genres"]))
print(movies.loc[0, "genres"])

<class 'list'>
[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 878, 'name': 'Science Fiction'}]


In [8]:
# 파싱이 잘 되었는지 head()로 다시 한번 확인합니다.
movies[["id", "title", "genres", "keywords"]].head()

,id,title,genres,keywords
0,19995,Avatar,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 1463, 'name': 'culture clash'}, {'id':..."
1,285,Pirates of the Caribbean: At World's End,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 270, 'name': 'ocean'}, {'id': 726, 'na..."
2,206647,Spectre,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 470, 'name': 'spy'}, {'id': 818, 'name..."
3,49026,The Dark Knight Rises,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","[{'id': 849, 'name': 'dc comics'}, {'id': 853,..."
4,49529,John Carter,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 818, 'name': 'based on novel'}, {'id':..."


In [9]:
# credits 데이터도 파싱이 잘 되었는지 확인합니다.
credits[["movie_id", "title", "cast", "crew"]].head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{'cast_id': 242, 'character': 'Jake Sully', '...","[{'credit_id': '52fe48009251416c750aca23', 'de..."
1,285,Pirates of the Caribbean: At World's End,"[{'cast_id': 4, 'character': 'Captain Jack Spa...","[{'credit_id': '52fe4232c3a36847f800b579', 'de..."
2,206647,Spectre,"[{'cast_id': 1, 'character': 'James Bond', 'cr...","[{'credit_id': '54805967c3a36829b5002c41', 'de..."
3,49026,The Dark Knight Rises,"[{'cast_id': 2, 'character': 'Bruce Wayne / Ba...","[{'credit_id': '52fe4781c3a36847f81398c3', 'de..."
4,49529,John Carter,"[{'cast_id': 5, 'character': 'John Carter', 'c...","[{'credit_id': '52fe479ac3a36847f813eaa3', 'de..."


## 4. crew에서 감독(Director) 정보 추출하기

`crew` 컬럼에는 감독, 촬영감독, 편집자 등 다양한 제작진 정보가 리스트로 들어있습니다.
각 사람은 `{'job': '직업명', 'name': '이름', ...}` 형태의 딕셔너리로 표현되어 있는데,
여기서 `job` 값이 `'Director'`인 사람만 골라서 감독 이름을 추출합니다.

In [10]:
# crew 리스트 하나(영화 한 편의 제작진 정보)를 입력받아서,
# 그 중 job이 'Director'인 사람의 이름을 반환하는 함수를 만듭니다.
def get_director(crew_list):
    # crew_list는 [{'job': 'Director', 'name': '홍길동', ...}, {'job': 'Editor', ...}, ...] 형태
    for member in crew_list:
        if member["job"] == "Director":
            return member["name"]
    # 감독 정보가 없는 경우 (데이터 결측) None을 반환합니다.
    return None

# crew 컬럼 전체에 위 함수를 적용해서, 새로운 'director' 컬럼을 만듭니다.
credits["director"] = credits["crew"].apply(get_director)

# 결과 확인: 영화 제목과 감독 이름만 뽑아서 head()로 확인합니다.
credits[["title", "director"]].head()

,title,director
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton


In [11]:
# 감독 정보가 없는(None) 영화가 몇 편인지 확인해봅니다.
missing_director_count = credits["director"].isna().sum()
print(f"감독 정보가 없는 영화 수: {missing_director_count}편")

감독 정보가 없는 영화 수: 30편


## 5. movies와 credits 합치기 (merge)

`movies`는 `id` 컬럼, `credits`는 `movie_id` 컬럼이 같은 영화를 가리키는 고유 식별자입니다.
두 데이터를 이 값 기준으로 합쳐서(merge), 하나의 데이터프레임 안에서
장르/줄거리/평점 정보와 출연진/감독 정보를 동시에 다룰 수 있게 만듭니다.

두 데이터 모두 `title` 컬럼이 있어서 merge 시 컬럼명이 겹치므로,
credits 쪽에서는 `movie_id`, `director`, `cast` 정도만 필요한 컬럼으로 추려서 합칩니다.

In [12]:
# credits에서 merge에 필요한 컬럼만 선택합니다.
# (title은 movies 쪽에도 있으므로 credits의 title은 제외하고, movie_id/cast/crew/director만 가져옵니다)
credits_subset = credits[["movie_id", "cast", "crew", "director"]]

# movies.id 와 credits.movie_id 를 기준으로 두 데이터를 합칩니다. (left join)
df = movies.merge(credits_subset, left_on="id", right_on="movie_id", how="left")

print(f"합쳐진 데이터 shape: {df.shape}")
df.head()

합쳐진 데이터 shape: (4803, 24)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew,director
0,237000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.avatarmovie.com/,19995,"[{'id': 1463, 'name': 'culture clash'}, {'id':...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{'name': 'Ingenious Film Partners', 'id': 289...",...,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{'cast_id': 242, 'character': 'Jake Sully', '...","[{'credit_id': '52fe48009251416c750aca23', 'de...",James Cameron
1,300000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",http://disney.go.com/disneypictures/pirates/,285,"[{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{'name': 'Walt Disney Pictures', 'id': 2}, {'...",...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{'cast_id': 4, 'character': 'Captain Jack Spa...","[{'credit_id': '52fe4232c3a36847f800b579', 'de...",Gore Verbinski
2,245000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{'id': 470, 'name': 'spy'}, {'id': 818, 'name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{'name': 'Columbia Pictures', 'id': 5}, {'nam...",...,"[{'iso_639_1': 'fr', 'name': 'Français'}, {'is...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{'cast_id': 1, 'character': 'James Bond', 'cr...","[{'credit_id': '54805967c3a36829b5002c41', 'de...",Sam Mendes
3,250000000,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",http://www.thedarkknightrises.com/,49026,"[{'id': 849, 'name': 'dc comics'}, {'id': 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{'name': 'Legendary Pictures', 'id': 923}, {'...",...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,"[{'cast_id': 2, 'character': 'Bruce Wayne / Ba...","[{'credit_id': '52fe4781c3a36847f81398c3', 'de...",Christopher Nolan
4,260000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://movies.disney.com/john-carter,49529,"[{'id': 818, 'name': 'based on novel'}, {'id':...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{'name': 'Walt Disney Pictures', 'id': 2}]",...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,"[{'cast_id': 5, 'character': 'John Carter', 'c...","[{'credit_id': '52fe479ac3a36847f813eaa3', 'de...",Andrew Stanton


In [13]:
# merge가 잘 되었는지, 제목/장르/감독/출연진 위주로 다시 확인합니다.
df[["id", "title", "genres", "director", "cast"]].head()

,id,title,genres,director,cast
0,19995,Avatar,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",James Cameron,"[{'cast_id': 242, 'character': 'Jake Sully', '..."
1,285,Pirates of the Caribbean: At World's End,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Gore Verbinski,"[{'cast_id': 4, 'character': 'Captain Jack Spa..."
2,206647,Spectre,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Sam Mendes,"[{'cast_id': 1, 'character': 'James Bond', 'cr..."
3,49026,The Dark Knight Rises,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",Christopher Nolan,"[{'cast_id': 2, 'character': 'Bruce Wayne / Ba..."
4,49529,John Carter,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Andrew Stanton,"[{'cast_id': 5, 'character': 'John Carter', 'c..."


## 6. 기본 탐색 (EDA)

합쳐진 데이터의 전체적인 상태를 확인합니다.
- 컬럼별 데이터 타입
- 결측치(비어있는 값) 개수
- 기본 통계 정보

In [14]:
# info()로 전체 컬럼 수, 데이터 타입, 결측치 여부(Non-Null Count)를 한눈에 확인합니다.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status                4803 non-n

In [15]:
# 컬럼별 결측치 개수를 확인합니다. (내림차순 정렬)
# 결측치가 많은 컬럼은 나중에 추천 로직에서 어떻게 처리할지 결정해야 합니다.
df.isna().sum().sort_values(ascending=False)

homepage                3091
tagline                  844
director                  30
overview                   3
runtime                    2
release_date               1
genres                     0
budget                     0
original_title             0
original_language          0
keywords                   0
id                         0
production_countries       0
revenue                    0
production_companies       0
popularity                 0
status                     0
spoken_languages           0
vote_average               0
title                      0
vote_count                 0
movie_id                   0
cast                       0
crew                       0
dtype: int64

In [16]:
# 숫자형 컬럼들의 기본 통계(평균, 최소/최대값, 분위수 등)를 확인합니다.
# 특히 vote_average(평점), vote_count(투표수)는 다음 단계(가중 평점 계산)에서 중요하게 쓰입니다.
df[["vote_average", "vote_count", "popularity", "runtime", "budget", "revenue"]].describe()

,vote_average,vote_count,popularity,runtime,budget,revenue
count,4803.000000,4803.000000,4803.000000,4801.000000,4.803000e+03,4.803000e+03
mean,6.092172,690.217989,21.492301,106.875859,2.904504e+07,8.226064e+07
std,1.194612,1234.585891,31.816650,22.611935,4.072239e+07,1.628571e+08
min,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
25%,5.600000,54.000000,4.668070,94.000000,7.900000e+05,0.000000e+00
50%,6.200000,235.000000,12.921594,103.000000,1.500000e+07,1.917000e+07
75%,6.800000,737.000000,28.313505,118.000000,4.000000e+07,9.291719e+07
max,10.000000,13752.000000,875.581305,338.000000,3.800000e+08,2.787965e+09


In [17]:
# 중복된 영화(id 기준)가 있는지 확인합니다. 0이면 중복이 없다는 뜻입니다.
duplicate_count = df["id"].duplicated().sum()
print(f"중복된 영화 id 개수: {duplicate_count}")

중복된 영화 id 개수: 0


In [18]:
# 최종 정리: 다음 단계(장르 벡터화)에서 주로 사용할 컬럼들만 모아서 눈으로 최종 확인합니다.
key_columns = ["id", "title", "genres", "keywords", "director", "cast", "overview", "vote_average", "vote_count"]
df[key_columns].head()

,id,title,genres,keywords,director,cast,overview,vote_average,vote_count
0,19995,Avatar,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 1463, 'name': 'culture clash'}, {'id':...",James Cameron,"[{'cast_id': 242, 'character': 'Jake Sully', '...","In the 22nd century, a paraplegic Marine is di...",7.2,11800
1,285,Pirates of the Caribbean: At World's End,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...",Gore Verbinski,"[{'cast_id': 4, 'character': 'Captain Jack Spa...","Captain Barbossa, long believed to be dead, ha...",6.9,4500
2,206647,Spectre,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 470, 'name': 'spy'}, {'id': 818, 'name...",Sam Mendes,"[{'cast_id': 1, 'character': 'James Bond', 'cr...",A cryptic message from Bond’s past sends him o...,6.3,4466
3,49026,The Dark Knight Rises,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","[{'id': 849, 'name': 'dc comics'}, {'id': 853,...",Christopher Nolan,"[{'cast_id': 2, 'character': 'Bruce Wayne / Ba...",Following the death of District Attorney Harve...,7.6,9106
4,49529,John Carter,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 818, 'name': 'based on novel'}, {'id':...",Andrew Stanton,"[{'cast_id': 5, 'character': 'John Carter', 'c...","John Carter is a war-weary, former military ca...",6.1,2124


## 정리

여기까지 진행한 내용:
1. 라이브러리 설치 확인
2. `movies.csv`, `credits.csv` 로딩
3. `genres`, `keywords`, `cast`, `crew`의 JSON 문자열을 실제 파이썬 객체로 파싱
4. `crew`에서 감독(director) 정보 추출
5. `movies`와 `credits`를 `id` 기준으로 merge → `df` 생성
6. 결측치 / 데이터 타입 등 기본 탐색(EDA)

다음 단계에서는 `genres`(및 keywords, director, cast)를 벡터화하고,
코사인 유사도(cosine similarity)를 계산해서 실제 추천 로직을 만들어보겠습니다.

## 7. 장르 텍스트 벡터화 (CountVectorizer)

컴퓨터는 `['Action', 'Adventure', 'Fantasy']` 같은 리스트를 그대로 이해하지 못합니다.
그래서 **장르 리스트를 숫자 벡터로 바꿔주는 작업**이 필요합니다.

### 진행 순서
1. `genres` 컬럼(딕셔너리 리스트)에서 장르 이름만 뽑아 공백으로 이어붙인 문자열을 만듭니다.
   - 예: `[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]` → `"Action Adventure"`
2. `CountVectorizer`로 이 문자열들을 벡터로 변환합니다.
   - `CountVectorizer`는 문장 안에 어떤 단어가 몇 번 등장하는지를 세어서 숫자 벡터로 만들어주는 도구입니다.
   - 예를 들어 전체 장르 단어 집합이 `[Action, Comedy, Drama]`라면,
     `"Action Drama"`라는 영화는 `[1, 0, 1]`이라는 벡터로 표현됩니다.

In [19]:
# genres 컬럼(딕셔너리 리스트)에서 'name' 값만 뽑아 공백으로 이어붙인 문자열 컬럼을 만듭니다.
# 예: [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}] -> "Action Adventure"
df["genres_literal"] = df["genres"].apply(lambda genre_list: " ".join([g["name"] for g in genre_list]))

# 결과 확인
df[["title", "genres", "genres_literal"]].head()

,title,genres,genres_literal
0,Avatar,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action Adventure Fantasy Science Fiction
1,Pirates of the Caribbean: At World's End,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Adventure Fantasy Action
2,Spectre,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action Adventure Crime
3,The Dark Knight Rises,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",Action Crime Drama Thriller
4,John Carter,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action Adventure Science Fiction


In [20]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer 옵션 설명
# - min_df=1: 최소 1개 영화에서라도 등장한 단어(장르)는 전부 포함시킵니다 (장르 종류가 많지 않으므로 제외할 필요 없음)
# - ngram_range=(1, 2): 단어 1개(uni-gram)뿐만 아니라 '연속된 단어 2개 조합'(bi-gram)도 함께 고려합니다.
#   예: "Action Adventure"에서 "Action", "Adventure" 뿐 아니라 "Action Adventure"라는 조합도 하나의 특성으로 봅니다.
#   → 장르의 '조합' 자체도 유사도 계산에 반영하기 위함입니다.
count_vect = CountVectorizer(min_df=1, ngram_range=(1, 2))

# fit_transform: 텍스트 데이터를 학습(fit)하면서 동시에 숫자 벡터로 변환(transform)합니다.
genre_mat = count_vect.fit_transform(df["genres_literal"])

# genre_mat은 (영화 개수, 전체 장르/장르조합 단어 개수) 크기의 희소 행렬(sparse matrix)입니다.
print(f"genre_mat shape: {genre_mat.shape}")

genre_mat shape: (4803, 276)


## 8. 코사인 유사도(Cosine Similarity) 행렬 계산

이제 각 영화가 숫자 벡터로 표현되었으니, **벡터끼리 얼마나 비슷한지**를 계산할 수 있습니다.

**코사인 유사도**는 두 벡터 사이의 '각도'가 얼마나 가까운지를 계산하는 방법입니다.
- 두 벡터의 방향이 완전히 같으면 → 유사도 1 (100% 비슷함)
- 두 벡터가 서로 관련이 없으면(직각) → 유사도 0
- 장르 벡터는 값이 음수가 될 수 없기 때문에, 코사인 유사도는 항상 0~1 사이 값을 가집니다.

`genre_mat`의 모든 영화 쌍(pair)에 대해 코사인 유사도를 계산하면,
`(영화 개수) x (영화 개수)` 크기의 정사각 행렬이 만들어집니다.
이 행렬의 `[i, j]` 값은 'i번째 영화'와 'j번째 영화'가 얼마나 비슷한지를 나타냅니다.

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

# genre_mat 자기 자신과의 코사인 유사도를 계산합니다.
# 결과: (영화 개수, 영화 개수) 크기의 정사각 행렬
genre_sim = cosine_similarity(genre_mat, genre_mat)

print(f"genre_sim shape: {genre_sim.shape}")

# 0번 영화(첫 번째 영화)와 다른 영화들 간의 유사도 값 중 앞부분 10개만 살펴봅니다.
print("0번 영화와 다른 영화들의 유사도 (앞 10개):")
print(genre_sim[0][:10])

genre_sim shape: (4803, 4803)
0번 영화와 다른 영화들의 유사도 (앞 10개):
[1.         0.59628479 0.4472136  0.12598816 0.75592895 0.59628479
 0.         0.75592895 0.4472136  0.74535599]


## 9. 영화 제목으로 유사 영화 추천 함수 만들기

`genre_sim` 행렬을 이용해서, **특정 영화와 가장 비슷한(유사도가 높은) 영화들**을 찾는 함수를 만듭니다.

### 아이디어
1. `genre_sim`의 각 행(row)을 유사도가 높은 순서로 정렬한 '인덱스 배열'을 미리 만들어둡니다.
   - 예를 들어 0번 영화 행을 정렬하면, "0번 영화와 가장 비슷한 영화의 인덱스, 두 번째로 비슷한 영화의 인덱스, ..." 순서로 나열됩니다.
2. 사용자가 영화 제목을 입력하면, 그 영화의 인덱스를 찾습니다.
3. 정렬된 인덱스 배열에서 상위 N개(자기 자신은 제외)를 뽑아 추천 목록으로 반환합니다.

In [22]:
# argsort()는 기본적으로 '오름차순(작은 값 -> 큰 값)' 인덱스를 반환합니다.
# 우리는 유사도가 '높은 순'으로 보고 싶으므로, [:, ::-1]로 각 행(row)을 뒤집어서 내림차순으로 만듭니다.
genre_sim_sorted_idx = genre_sim.argsort()[:, ::-1]

# 0번 영화와 가장 비슷한 영화들의 인덱스를 앞 5개만 확인해봅니다.
# (맨 앞은 자기 자신(0번)일 가능성이 높습니다 - 자기 자신과의 유사도가 1로 가장 높기 때문)
print(genre_sim_sorted_idx[0][:5])

[   0   46 3494  870   14]


In [23]:
def find_sim_movie(df, sorted_idx, title_name, top_n=10):
    """
    장르 유사도만을 기준으로, 입력한 영화 제목과 가장 비슷한 영화 top_n개를 반환합니다.

    Parameters
    ----------
    df : DataFrame
        영화 정보가 담긴 데이터프레임 (title, genres_literal 등 포함)
    sorted_idx : ndarray
        영화별로 유사도가 높은 순서로 정렬된 인덱스 배열 (genre_sim_sorted_idx)
    title_name : str
        기준이 되는 영화 제목
    top_n : int
        추천받고 싶은 영화 개수
    """
    # 1) 입력한 제목과 정확히 일치하는 영화의 행(row)을 찾습니다.
    title_movie = df[df["title"] == title_name]

    if title_movie.empty:
        print(f"'{title_name}' 영화를 데이터에서 찾을 수 없습니다.")
        return None

    # 2) 해당 영화의 인덱스(위치 번호)를 가져옵니다.
    title_index = title_movie.index.values

    # 3) 그 영화 행의 정렬된 유사 영화 인덱스 중, 맨 앞(자기 자신)을 제외하고 top_n개를 가져옵니다.
    similar_indexes = sorted_idx[title_index, 1:(top_n + 1)]

    # similar_indexes는 2차원 배열(예: [[3, 7, 15, ...]])이므로 1차원으로 펼쳐줍니다.
    similar_indexes = similar_indexes.reshape(-1)

    # 4) 인덱스에 해당하는 영화 정보를 반환합니다.
    return df.iloc[similar_indexes][["title", "genres_literal", "vote_average", "vote_count"]]

In [24]:
# "The Dark Knight Rises"와 장르가 비슷한 영화 10편을 추천받아봅니다.
similar_movies = find_sim_movie(df, genre_sim_sorted_idx, "The Dark Knight Rises", top_n=10)
similar_movies

,title,genres_literal,vote_average,vote_count
2609,Dark Blue,Action Crime Drama Thriller,6.5,85
2218,Death Sentence,Action Crime Drama Thriller,6.5,297
2195,Armored,Action Crime Drama Thriller,5.5,208
4408,Jimmy and Judy,Action Crime Drama Thriller,5.4,8
761,Righteous Kill,Action Crime Drama Thriller,5.9,375
762,Mercury Rising,Action Crime Drama Thriller,6.0,368
3073,Romeo Is Bleeding,Action Crime Drama Thriller,5.7,36
1052,Training Day,Action Crime Drama Thriller,7.3,1634
2154,Street Kings,Action Crime Drama Thriller,6.3,363
629,Need for Speed,Action Crime Drama Thriller,6.1,1520


## 정리 및 다음 단계 예고

여기까지 진행한 내용:
7. `genres`를 `CountVectorizer`로 벡터화 (`genre_mat`)
8. 코사인 유사도 행렬 계산 (`genre_sim`)
9. 영화 제목 입력 → 장르 유사도 기반 추천 함수 구현 (`find_sim_movie`)

결과를 보면 장르는 비슷하지만 `vote_average`(평점)가 낮거나 `vote_count`(투표 수)가 매우 적은 영화도
추천 목록에 섞여 있는 것을 확인할 수 있습니다. 즉, **"장르만 비슷하면 무조건 좋은 추천"은 아니라는 문제**가 있습니다.

다음 단계에서는 **가중 평점(Weighted Rating)** 을 함께 반영해서,
장르도 비슷하면서 평점 신뢰도도 높은 영화를 우선적으로 추천하도록 로직을 개선하겠습니다.

## 10. 가중 평점(Weighted Rating) 반영하기

앞서 확인했듯이, `vote_count`(투표 수)가 아주 적은 영화는 `vote_average`(평점)를 그대로 믿기 어렵습니다.
예를 들어 투표 5명이 전부 만점을 줘서 평점이 10.0인 영화와, 10,000명이 투표해서 평점이 8.5인 영화가 있다면
후자가 더 신뢰할 수 있는 평점이겠죠.

이 문제를 보정하기 위해 IMDB에서 사용하는 **가중 평점 공식**을 사용합니다.

```
가중 평점 = (v / (v + m)) × R + (m / (v + m)) × C

v = 해당 영화의 투표 수 (vote_count)
m = 추천 리스트에 포함되기 위한 최소 투표 수 기준치 (전체 데이터의 특정 분위수로 설정)
R = 해당 영화의 평균 평점 (vote_average)
C = 전체 영화의 평균 평점
```

**쉽게 풀어서 설명하면**: 투표 수(v)가 기준치(m)보다 훨씬 많으면 그 영화의 실제 평점(R)을 거의 그대로 신뢰하고,
반대로 투표 수가 적으면 전체 평균(C) 쪽으로 점수를 끌어당겨서 "너무 적은 사람이 매긴 극단적인 평점"의 영향을 줄입니다.

In [25]:
# C: 전체 영화의 평균 평점
C = df["vote_average"].mean()

# m: 최소 투표 수 기준치.
# 전체 영화 중 투표 수(vote_count) 상위 40%(=60번째 백분위수 이상)에 해당하는 영화만
# '평점을 신뢰할 만하다'고 보고, 그 경계값을 m으로 사용합니다.
percentile = 0.6
m = df["vote_count"].quantile(percentile)

print(f"전체 평균 평점 (C): {C:.3f}")
print(f"최소 투표 수 기준치 (m): {m:.1f}")

전체 평균 평점 (C): 6.092
최소 투표 수 기준치 (m): 370.2


In [26]:
def weighted_vote_average(record):
    v = record["vote_count"]
    R = record["vote_average"]
    # 위에서 설명한 공식을 그대로 적용합니다.
    return (v / (v + m)) * R + (m / (v + m)) * C

# 영화(행)마다 가중 평점을 계산해서 새로운 컬럼으로 추가합니다.
# axis=1: 한 행씩(row-wise) 함수를 적용하라는 의미입니다.
df["weighted_vote"] = df.apply(weighted_vote_average, axis=1)

# 원래 평점(vote_average)과 가중 평점(weighted_vote)을 비교해봅니다.
# 투표 수가 적은 영화일수록 두 값의 차이가 크게 나는 것을 확인할 수 있습니다.
df[["title", "vote_average", "vote_count", "weighted_vote"]].sort_values("weighted_vote", ascending=False).head(10)

,title,vote_average,vote_count,weighted_vote
1881,The Shawshank Redemption,8.5,8205,8.396052
3337,The Godfather,8.4,5893,8.263591
662,Fight Club,8.3,9413,8.216455
3232,Pulp Fiction,8.3,8428,8.207102
65,The Dark Knight,8.2,12002,8.136930
1818,Schindler's List,8.3,4329,8.126069
3865,Whiplash,8.3,4254,8.123248
809,Forrest Gump,8.2,7927,8.105954
2294,Spirited Away,8.3,3840,8.105867
2731,The Godfather: Part II,8.3,3338,8.079586


## 11. 가중 평점을 반영한 추천 함수

이제 두 가지를 함께 고려하는 새로운 추천 함수를 만듭니다.

### 아이디어
1. 장르 유사도 순으로 정렬된 인덱스에서, 최종적으로 원하는 개수(`top_n`)보다 **더 많은 후보**(`top_n * 2`)를 우선 뽑습니다.
   - 유사도 상위권 후보들을 넉넉하게 확보해둔다는 의미입니다.
2. 그 후보들 중에서 `weighted_vote`(가중 평점)가 높은 순서로 다시 정렬해서 최종 top_n개를 선택합니다.

즉, **"장르가 비슷한 영화들 중에서, 평점 신뢰도가 높은 영화를 우선 추천"** 하는 방식입니다.

In [27]:
def find_sim_movie_weighted(df, sorted_idx, title_name, top_n=10):
    """
    장르 유사도 + 가중 평점을 함께 고려해서, 입력한 영화 제목과 비슷한 영화 top_n개를 반환합니다.
    """
    title_movie = df[df["title"] == title_name]

    if title_movie.empty:
        print(f"'{title_name}' 영화를 데이터에서 찾을 수 없습니다.")
        return None

    title_index = title_movie.index.values

    # 1) 장르 유사도 상위 (top_n * 2)개 후보를 넉넉하게 뽑습니다. (자기 자신 포함)
    candidate_indexes = sorted_idx[title_index, :(top_n * 2)]
    candidate_indexes = candidate_indexes.reshape(-1)

    # 2) 후보 중에서 자기 자신(입력한 영화)은 제외합니다.
    candidate_indexes = candidate_indexes[candidate_indexes != title_index[0]]

    # 3) 후보들 중 가중 평점(weighted_vote)이 높은 순으로 정렬해서 상위 top_n개만 선택합니다.
    candidates = df.iloc[candidate_indexes]
    result = candidates.sort_values("weighted_vote", ascending=False).head(top_n)

    return result[["title", "genres_literal", "vote_average", "vote_count", "weighted_vote"]]

In [28]:
# "The Dark Knight Rises" 기준, 가중 평점을 반영한 추천 결과를 확인합니다.
weighted_similar_movies = find_sim_movie_weighted(df, genre_sim_sorted_idx, "The Dark Knight Rises", top_n=10)
weighted_similar_movies

,title,genres_literal,vote_average,vote_count,weighted_vote
1052,Training Day,Action Crime Drama Thriller,7.3,1634,7.076899
2435,Running Scared,Action Crime Drama Thriller,7.0,331,6.520710
3966,Point Blank,Action Crime Drama Thriller,7.1,95,6.297983
2218,Death Sentence,Action Crime Drama Thriller,6.5,297,6.273714
2154,Street Kings,Action Crime Drama Thriller,6.3,363,6.195065
2609,Dark Blue,Action Crime Drama Thriller,6.5,85,6.168326
3180,The Way of the Gun,Action Crime Drama Thriller,6.4,100,6.157639
405,The Fast and the Furious: Tokyo Drift,Action Crime Drama Thriller,6.1,1705,6.098603
629,Need for Speed,Action Crime Drama Thriller,6.1,1520,6.098467
4230,Killing Zoe,Action Crime Drama Thriller,6.1,111,6.093977


## 12. Before / After 비교

- **Before**: 순수 장르 유사도만으로 뽑은 추천 목록 (`find_sim_movie`)
- **After**: 장르 유사도 + 가중 평점을 함께 반영한 추천 목록 (`find_sim_movie_weighted`)

두 목록을 나란히 비교해서, 가중 평점을 반영했을 때 순위가 어떻게 바뀌는지(또는 목록에서
빠지거나 새로 들어오는 영화가 무엇인지) 확인합니다. 이 로직은 나중에 웹 UI의
Before/After 비교 화면에도 그대로 사용할 예정입니다.

In [29]:
def compare_before_after(df, sorted_idx, title_name, top_n=10):
    """
    Before(순수 유사도)와 After(가중 평점 반영) 추천 목록을 나란히 비교하는 표를 만듭니다.
    순위가 바뀐 영화는 순위 변동(rank_change)을, 한쪽에만 있는 영화는 'New'/'Dropped'로 표시합니다.
    """
    before = find_sim_movie(df, sorted_idx, title_name, top_n=top_n).reset_index(drop=True)
    after = find_sim_movie_weighted(df, sorted_idx, title_name, top_n=top_n).reset_index(drop=True)

    # 순위는 표에서의 위치(0번째 = 1등)로 정의합니다.
    before_rank = {title: rank + 1 for rank, title in enumerate(before["title"])}
    after_rank = {title: rank + 1 for rank, title in enumerate(after["title"])}

    # Before, After 어느 한쪽에라도 등장한 영화 전체 목록 (중복 제거)
    all_titles = list(dict.fromkeys(list(before["title"]) + list(after["title"])))

    rows = []
    for title in all_titles:
        b_rank = before_rank.get(title)  # Before 순위 (없으면 None)
        a_rank = after_rank.get(title)   # After 순위 (없으면 None)

        if b_rank is not None and a_rank is not None:
            # 두 목록 모두에 있는 경우: 순위 변동을 계산합니다. (양수 = 순위 상승)
            change = b_rank - a_rank
            if change > 0:
                status = f"⬆️ {change}단계 상승"
            elif change < 0:
                status = f"⬇️ {abs(change)}단계 하락"
            else:
                status = "변동 없음"
        elif a_rank is not None:
            status = "🆕 새로 추천됨 (After에만 있음)"
        else:
            status = "❌ 제외됨 (Before에만 있음)"

        rows.append({"title": title, "before_rank": b_rank, "after_rank": a_rank, "status": status})

    return pd.DataFrame(rows)

comparison_table = compare_before_after(df, genre_sim_sorted_idx, "The Dark Knight Rises", top_n=10)
comparison_table

,title,before_rank,after_rank,status
0,Dark Blue,1.0,6.0,⬇️ 5단계 하락
1,Death Sentence,2.0,4.0,⬇️ 2단계 하락
2,Armored,3.0,NaN,❌ 제외됨 (Before에만 있음)
3,Jimmy and Judy,4.0,NaN,❌ 제외됨 (Before에만 있음)
4,Righteous Kill,5.0,NaN,❌ 제외됨 (Before에만 있음)
5,Mercury Rising,6.0,NaN,❌ 제외됨 (Before에만 있음)
6,Romeo Is Bleeding,7.0,NaN,❌ 제외됨 (Before에만 있음)
7,Training Day,8.0,1.0,⬆️ 7단계 상승
8,Street Kings,9.0,5.0,⬆️ 4단계 상승
9,Need for Speed,10.0,9.0,⬆️ 1단계 상승


## 정리 및 다음 단계 예고

여기까지 진행한 내용:
10. IMDB 방식의 가중 평점(`weighted_vote`) 계산
11. 장르 유사도 + 가중 평점을 함께 반영한 추천 함수 `find_sim_movie_weighted` 구현
12. Before(순수 유사도) / After(가중 평점 반영) 추천 결과 비교표 구현

다음 단계에서는:
- 웹 UI(`app/recommender_ui.html`)에서 사용할 수 있도록 추천 결과와 포스터 정보를 **JSON으로 export**
- TMDB API로 포스터 이미지 경로(`poster_path`) 조회
- 정적 웹 UI 제작 (검색 → 추천 카드 그리드 → 설명 패널 → Before/After 비교 화면)

을 진행할 예정입니다.

## 13. TMDB API로 포스터 이미지 경로 조회하기

`tmdb_5000_movies.csv`에는 포스터 이미지 경로(`poster_path`)가 들어있지 않습니다.
그래서 TMDB(The Movie Database)의 API를 호출해서, 영화 id별로 포스터 경로를 따로 조회합니다.

- API 키는 `.env` 파일의 `TMDB_API_KEY`에서 안전하게 불러옵니다 (코드에 직접 적지 않음, git에도 올라가지 않음)
- 영화가 4,800편이 넘기 때문에, 하나씩 순서대로 요청하면 시간이 오래 걸립니다.
  → `concurrent.futures.ThreadPoolExecutor`로 **여러 요청을 동시에** 보내서 속도를 높입니다.
- 포스터가 없는 영화(TMDB에 없거나 삭제된 영화 등)는 `None`으로 남겨두고, 웹 UI에서는 기본 이미지로 대체할 예정입니다.

In [30]:
import os
import concurrent.futures
from dotenv import load_dotenv

# 프로젝트 루트에 있는 .env 파일을 읽어옵니다. (notebook/ 폴더 기준 상위 폴더)
load_dotenv("../.env")
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not TMDB_API_KEY:
    raise ValueError("TMDB_API_KEY를 찾을 수 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.")

# TMDB 포스터 이미지는 이 기본 URL 뒤에 poster_path를 붙이면 바로 접근할 수 있습니다.
TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"

print("TMDB API 키 로딩 완료 ✅")

TMDB API 키 로딩 완료 ✅


In [31]:
def fetch_poster_path(movie_id):
    """
    TMDB API로 영화 한 편의 상세 정보를 조회해서 poster_path만 뽑아 반환합니다.
    요청이 실패하거나 포스터 정보가 없으면 None을 반환합니다.
    """
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    try:
        response = requests.get(url, params={"api_key": TMDB_API_KEY}, timeout=5)
        if response.status_code == 200:
            return movie_id, response.json().get("poster_path")
    except requests.RequestException:
        pass
    return movie_id, None


movie_ids = df["id"].tolist()
poster_path_by_id = {}

# ThreadPoolExecutor: 여러 개의 HTTP 요청을 동시에(병렬로) 보내주는 도구입니다.
# max_workers=20 -> 최대 20개의 요청을 동시에 진행합니다.
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(fetch_poster_path, movie_id) for movie_id in movie_ids]

    completed = 0
    for future in concurrent.futures.as_completed(futures):
        movie_id, poster_path = future.result()
        poster_path_by_id[movie_id] = poster_path
        completed += 1
        # 진행 상황을 500개 단위로 출력합니다.
        if completed % 500 == 0:
            print(f"{completed}/{len(movie_ids)}개 조회 완료")

print(f"포스터 경로 조회 완료 ✅ (전체 {len(movie_ids)}개)")

500/4803개 조회 완료


1000/4803개 조회 완료


1500/4803개 조회 완료


2000/4803개 조회 완료


2500/4803개 조회 완료


3000/4803개 조회 완료


3500/4803개 조회 완료


4000/4803개 조회 완료


4500/4803개 조회 완료


포스터 경로 조회 완료 ✅ (전체 4803개)


In [32]:
# 조회한 poster_path를 df에 매핑하고, 실제로 접근 가능한 전체 이미지 URL도 만들어둡니다.
df["poster_path"] = df["id"].map(poster_path_by_id)
df["poster_url"] = df["poster_path"].apply(
    lambda path: f"{TMDB_IMAGE_BASE_URL}{path}" if pd.notna(path) else None
)

missing_poster_count = df["poster_url"].isna().sum()
print(f"포스터를 찾지 못한 영화 수: {missing_poster_count}편 / 전체 {len(df)}편")

df[["title", "poster_path", "poster_url"]].head()

포스터를 찾지 못한 영화 수: 20편 / 전체 4803편


,title,poster_path,poster_url
0,Avatar,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,https://image.tmdb.org/t/p/w500/gKY6q7SjCkAU6F...
1,Pirates of the Caribbean: At World's End,/jGWpG4YhpQwVmjyHEGkxEkeRf0S.jpg,https://image.tmdb.org/t/p/w500/jGWpG4YhpQwVmj...
2,Spectre,/zj8ongFhtWNsVlfjOGo8pSr7PQg.jpg,https://image.tmdb.org/t/p/w500/zj8ongFhtWNsVl...
3,The Dark Knight Rises,/hr0L2aueqlP2BYUblTTjmtn0hw4.jpg,https://image.tmdb.org/t/p/w500/hr0L2aueqlP2BY...
4,John Carter,/lCxz1Yus07QCQQCb6I0Dr3Lmqpx.jpg,https://image.tmdb.org/t/p/w500/lCxz1Yus07QCQQ...


## 14. 웹 UI용 데이터 export (`app/data.json`)

`app/recommender_ui.html`은 **파이썬 없이, 순수 브라우저(JavaScript)만으로 동작하는 정적 페이지**입니다.
따라서 코사인 유사도 계산 같은 무거운 작업을 브라우저에서 다시 할 수 없고,
노트북에서 이미 계산해둔 결과를 **미리 계산된 형태(JSON)** 로 저장해서 웹 페이지가 그냥 읽어 쓰게 만듭니다.

### 무엇을 저장할까?
전체 코사인 유사도 행렬(4,800 x 4,800)을 통째로 저장하면 용량이 너무 커지므로,
**영화별로 미리 뽑아둔 추천 목록(top 20)만** 저장합니다.

- `movies`: 영화 id를 key로 하는 딕셔너리. 제목/장르/감독/줄거리/평점/포스터 URL + 추천 목록(before/after) 포함
- `search_index`: 검색창 자동완성에 사용할 (id, title) 목록

이렇게 하면 웹 페이지는 이 JSON 파일 하나만 불러와서, 검색 → 추천 카드 표시 → Before/After 비교까지
서버 없이 전부 처리할 수 있습니다.

In [33]:
import json

# 영화당 몇 개의 추천 목록을 저장해둘지 정합니다.
# (웹 UI에서 "상위 10개만 보여줘" 처럼 더 적게 쓰더라도, 넉넉하게 20개까지 미리 저장해둡니다)
TOP_N_EXPORT = 20

movies_export = {}
n_movies = len(df)

for i in range(n_movies):
    row = df.iloc[i]
    movie_id = int(row["id"])

    # --- Before: 순수 장르 유사도 상위 TOP_N_EXPORT개 (자기 자신은 제외) ---
    before_indexes = genre_sim_sorted_idx[i, 1:TOP_N_EXPORT + 1]
    before_list = [
        {
            "id": int(df.iloc[j]["id"]),
            "similarity": round(float(genre_sim[i, j]), 4),
        }
        for j in before_indexes
    ]

    # --- After: 유사도 상위 (TOP_N_EXPORT*2)개 후보 중, 가중 평점 순으로 재정렬한 TOP_N_EXPORT개 ---
    candidate_indexes = genre_sim_sorted_idx[i, :TOP_N_EXPORT * 2]
    candidate_indexes = candidate_indexes[candidate_indexes != i]  # 자기 자신 제외

    candidates = df.iloc[candidate_indexes].copy()
    candidates["similarity"] = [genre_sim[i, j] for j in candidate_indexes]
    candidates = candidates.sort_values("weighted_vote", ascending=False).head(TOP_N_EXPORT)

    after_list = [
        {
            "id": int(candidate_row["id"]),
            "similarity": round(float(candidate_row["similarity"]), 4),
            "weighted_vote": round(float(candidate_row["weighted_vote"]), 3),
        }
        for _, candidate_row in candidates.iterrows()
    ]

    # --- 영화 한 편의 정보를 딕셔너리로 정리 ---
    movies_export[str(movie_id)] = {
        "id": movie_id,
        "title": row["title"],
        "genres": [g["name"] for g in row["genres"]],
        "director": row["director"] if pd.notna(row["director"]) else None,
        "overview": row["overview"] if pd.notna(row["overview"]) else "",
        "release_date": row["release_date"] if pd.notna(row["release_date"]) else None,
        "vote_average": round(float(row["vote_average"]), 2),
        "vote_count": int(row["vote_count"]),
        "weighted_vote": round(float(row["weighted_vote"]), 3),
        "poster_url": row["poster_url"] if pd.notna(row["poster_url"]) else None,
        "recommendations": {
            "before": before_list,
            "after": after_list,
        },
    }

print(f"{len(movies_export)}개 영화의 데이터를 정리했습니다 ✅")

4803개 영화의 데이터를 정리했습니다 ✅


In [34]:
# 검색창 자동완성에 사용할 (id, title) 목록. 제목 가나다(알파벳)순으로 정렬해둡니다.
search_index = (
    df[["id", "title"]]
    .sort_values("title")
    .apply(lambda r: {"id": int(r["id"]), "title": r["title"]}, axis=1)
    .tolist()
)

# 최종 export 데이터
export_data = {
    "meta": {
        "movie_count": len(movies_export),
        "top_n": TOP_N_EXPORT,
        "weighted_rating": {"C": round(float(C), 3), "m": round(float(m), 1)},
    },
    "search_index": search_index,
    "movies": movies_export,
}

# notebook/ 폴더 기준 상위 폴더의 app/ 폴더에 저장합니다.
OUTPUT_PATH = "../app/data.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    # ensure_ascii=False: 한글이 유니코드 이스케이프(\uXXXX)가 아니라 그대로 저장되게 합니다.
    json.dump(export_data, f, ensure_ascii=False)

file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"{OUTPUT_PATH} 저장 완료 ✅ (파일 크기: {file_size_mb:.2f} MB)")

../app/data.json 저장 완료 ✅ (파일 크기: 11.81 MB)


## 정리

이번 노트북에서 진행한 전체 내용:
1~6. 데이터 로딩 및 가공 (라이브러리 확인, CSV 로딩, JSON 파싱, 감독 추출, merge, EDA)
7~9. 장르 벡터화 → 코사인 유사도 계산 → 유사 영화 추천 함수
10~12. 가중 평점 반영 → Before/After 비교
13~14. TMDB 포스터 조회 → 웹 UI용 `app/data.json` export

다음 단계는 `app/data.json`을 읽어서 화면에 보여주는 정적 웹 페이지(`app/recommender_ui.html`) 제작입니다.